In [25]:
# XL Net testing
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import XLNetTokenizer, XLNetForSequenceClassification
from huggingface_hub import notebook_login
notebook_login()

In [4]:
# Load dataset
asap_train = load_dataset(
    "nlpscu/Analyzing-Demographic-Biases", 
    data_files={"train": "ASAP/ASAP_2_Final_github_train.csv"}
)["train"].to_pandas()
asap_test = load_dataset(
    "nlpscu/Analyzing-Demographic-Biases", 
    data_files={"test": "ASAP/ASAP_2_Final_github_test.csv"}
)["test"].to_pandas()

README.md: 0.00B [00:00, ?B/s]

ASAP/ASAP_2_Final_github_train.csv:   0%|          | 0.00/46.5M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

ASAP/ASAP_2_Final_github_test.csv:   0%|          | 0.00/18.9M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [ ]:
persuade_train = pd.read_csv("hf://datasets/nlpscu/Analyzing-Demographic-Biases/PERSUADE/persuade_corpus_2.0_train.csv", low_memory=False)
persuade_test = pd.read_csv("hf://datasets/nlpscu/Analyzing-Demographic-Biases/PERSUADE/persuade_corpus_2.0_test.csv", low_memory=False)
from sklearn.model_selection import train_test_split
pers_train_df, pers_dev_df = train_test_split(persuade_train, test_size=0.1, random_state=42)
hf_pers_train = Dataset.from_pandas(pers_train_df)
hf_pers_dev = Dataset.from_pandas(pers_dev_df)
hf_pers_test = Dataset.from_pandas(persuade_test)

In [17]:
# Tokenize PERSUADE
tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")

def tokenize_function(dataset):
    return tokenizer(dataset["full_text"], padding="max_length", truncation=True, max_length=2048)

print("Tokenizing training data...")
tokenized_pers_train = hf_pers_train.map(tokenize_function, batched = True)
print("Tokenizing development data...")
tokenized_pers_dev = hf_pers_dev.map(tokenize_function, batched = True)
print("Tokenizing test data...")
tokenized_pers_test = hf_pers_test.map(tokenize_function, batched = True)

Tokenizing training data...


Map:   0%|          | 0/155939 [00:00<?, ? examples/s]

Tokenizing development data...


Map:   0%|          | 0/17327 [00:00<?, ? examples/s]

Tokenizing test data...


Map:   0%|          | 0/112117 [00:00<?, ? examples/s]

In [18]:
from torch.utils.data import DataLoader

batch_size = 8
train_dataloader = DataLoader(tokenized_pers_train, shuffle=True, batch_size=batch_size)
dev_dataloader = DataLoader(tokenized_pers_dev, batch_size=batch_size)
test_dataloader = DataLoader(tokenized_pers_test, batch_size=batch_size)

In [19]:
model_XLNet = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_XLNet.to(device)

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

XLNetForSequenceClassification LOAD REPORT from: xlnet-base-cased
Key                             | Status     | 
--------------------------------+------------+-
lm_loss.weight                  | UNEXPECTED | 
lm_loss.bias                    | UNEXPECTED | 
logits_proj.bias                | MISSING    | 
logits_proj.weight              | MISSING    | 
sequence_summary.summary.bias   | MISSING    | 
sequence_summary.summary.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


XLNetForSequenceClassification(
  (transformer): XLNetModel(
    (word_embedding): Embedding(32000, 768)
    (layer): ModuleList(
      (0-11): 12 x XLNetLayer(
        (rel_attn): XLNetRelativeAttention(
          (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): XLNetFeedForward(
          (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (layer_1): Linear(in_features=768, out_features=3072, bias=True)
          (layer_2): Linear(in_features=3072, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (activation_function): GELUActivation()
        )
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (sequence_summary): XLNetSequenceSummary(
    (summary): Linear(in_features=768, out_features=768, bias=True)
    (activation): Tanh()
    (first_dropout): Identity()
    

model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

In [24]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
learning_rate = 5e-6
optimizer = AdamW(model_XLNet.parameters(), lr=learning_rate)
epochs = 20
total_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
# Training loop
best_dev_loss = float('inf')
best_model_state = None
print("Start Training:")

for epoch in range(epochs): # epochs = 20
    model_XLNet.train()
    total_train_loss = 0
    
    # Training Pass
    for batch in train_dataloader:
        # Move batch to device (GPU/CPU)
        batch = {k: v.to(device) for k, v in batch.items()}
        
        # Clear previous gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model_XLNet(**batch)
        loss = outputs.loss
        total_train_loss += loss.item()
        
        # Backward pass & optimize
        loss.backward()
        optimizer.step()
        scheduler.step()
        
    avg_train_loss = total_train_loss / len(train_dataloader)
    
    # Dev Set Evaluation (Early Stopping Check)
    model_XLNet.eval()
    total_dev_loss = 0
    
    with torch.no_grad():
        for batch in dev_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model_XLNet(**batch)
            loss = outputs.loss
            total_dev_loss += loss.item()
            
    avg_dev_loss = total_dev_loss / len(dev_dataloader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Dev Loss: {avg_dev_loss:.4f}")
    
    # Save the model if it's the best one we've seen so far
    if avg_dev_loss < best_dev_loss:
        best_dev_loss = avg_dev_loss
        # Store the weights of the best performing model
        best_model_state = model_XLNet.state_dict()

# Load the best weights back into the model before moving to the test set
print("Training complete. Loading best model weights:")
model_XLNet.load_state_dict(best_model_state)

Start Training:


TypeError: must be real number, not NoneType